# INN Hotels Project

## Context

A significant number of hotel bookings are called-off due to cancellations or no-shows. The typical reasons for cancellations include change of plans, scheduling conflicts, etc. This is often made easier by the option to do so free of charge or preferably at a low cost which is beneficial to hotel guests but it is a less desirable and possibly revenue-diminishing factor for hotels to deal with. Such losses are particularly high on last-minute cancellations.

The new technologies involving online booking channels have dramatically changed customers’ booking possibilities and behavior. This adds a further dimension to the challenge of how hotels handle cancellations, which are no longer limited to traditional booking and guest characteristics.

The cancellation of bookings impact a hotel on various fronts:
* Loss of resources (revenue) when the hotel cannot resell the room.
* Additional costs of distribution channels by increasing commissions or paying for publicity to help sell these rooms.
* Lowering prices last minute, so the hotel can resell a room, resulting in reducing the profit margin.
* Human resources to make arrangements for the guests.

## Objective
The increasing number of cancellations calls for a Machine Learning based solution that can help in predicting which booking is likely to be canceled. INN Hotels Group has a chain of hotels in Portugal, they are facing problems with the high number of booking cancellations and have reached out to your firm for data-driven solutions. You as a data scientist have to analyze the data provided to find which factors have a high influence on booking cancellations, build a predictive model that can predict which booking is going to be canceled in advance, and help in formulating profitable policies for cancellations and refunds.

## Data Description
The data contains the different attributes of customers' booking details. The detailed data dictionary is given below.


**Data Dictionary**

* Booking_ID: unique identifier of each booking
* no_of_adults: Number of adults
* no_of_children: Number of Children
* no_of_weekend_nights: Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel
* no_of_week_nights: Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel
* type_of_meal_plan: Type of meal plan booked by the customer:
    * Not Selected – No meal plan selected
    * Meal Plan 1 – Breakfast
    * Meal Plan 2 – Half board (breakfast and one other meal)
    * Meal Plan 3 – Full board (breakfast, lunch, and dinner)
* required_car_parking_space: Does the customer require a car parking space? (0 - No, 1- Yes)
* room_type_reserved: Type of room reserved by the customer. The values are ciphered (encoded) by INN Hotels.
* lead_time: Number of days between the date of booking and the arrival date
* arrival_year: Year of arrival date
* arrival_month: Month of arrival date
* arrival_date: Date of the month
* market_segment_type: Market segment designation.
* repeated_guest: Is the customer a repeated guest? (0 - No, 1- Yes)
* no_of_previous_cancellations: Number of previous bookings that were canceled by the customer prior to the current booking
* no_of_previous_bookings_not_canceled: Number of previous bookings not canceled by the customer prior to the current booking
* avg_price_per_room: Average price per day of the reservation; prices of the rooms are dynamic. (in euros)
* no_of_special_requests: Total number of special requests made by the customer (e.g. high floor, view from the room, etc)
* booking_status: Flag indicating if the booking was canceled or not.

## Importing necessary libraries and data

In [1]:
# Installing the libraries with the specified version.
!pip install pandas==1.5.3 numpy==1.25.2 matplotlib==3.7.1 seaborn==0.13.1 scikit-learn==1.2.2 statsmodels==0.14.1 -q --user

**Note**: *After running the above cell, kindly restart the notebook kernel and run all cells sequentially from the start again.*

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import statsmodels.api as sm

# Loading the data
df = pd.read_csv('INNHotelsGroup.csv')

# Displaying the first few rows of the dataset
df.head()


## Data Overview

- Observations
- Sanity checks

In [ ]:
# Data Overview and Sanity Checks
# Basic statistics
basic_stats = df.describe(include='all')
print(basic_stats)

# Data types and missing values
data_info = df.info()
print(data_info)

# Checking for missing values
missing_values = df.isnull().sum()
print(missing_values)

## Exploratory Data Analysis (EDA)

- EDA is an important part of any project involving data.
- It is important to investigate and understand the data better before building a model with it.
- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
- A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Leading Questions**:
1. What are the busiest months in the hotel?
2. Which market segment do most of the guests come from?
3. Hotel rates are dynamic and change according to demand and customer demographics. What are the differences in room prices in different market segments?
4. What percentage of bookings are canceled?
5. Repeating guests are the guests who stay in the hotel often and are important to brand equity. What percentage of repeating guests cancel?
6. Many guests have special requirements when booking a hotel room. Do these requirements affect booking cancellation?

In [ ]:
# Exploratory Data Analysis (EDA)
# Distribution of Booking Status
plt.figure(figsize=(8, 6))
sns.countplot(x='booking_status', data=df)
plt.title('Distribution of Booking Status')
plt.xlabel('Booking Status')
plt.ylabel('Count')
plt.show()

# Busiest months
plt.figure(figsize=(10, 6))
sns.countplot(x='arrival_month', data=df, order=range(1, 13))
plt.title('Number of Bookings by Month')
plt.xlabel('Month')
plt.ylabel('Number of Bookings')
plt.show()

# Market segment distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='market_segment_type', data=df)
plt.title('Market Segment Distribution')
plt.xlabel('Market Segment')
plt.ylabel('Number of Bookings')
plt.show()

# Room prices by market segment
plt.figure(figsize=(10, 6))
sns.boxplot(x='market_segment_type', y='avg_price_per_room', data=df)
plt.title('Room Prices by Market Segment')
plt.xlabel('Market Segment')
plt.ylabel('Average Price per Room')
plt.show()

# Percentage of cancellations
cancelled_percentage = df['booking_status'].value_counts(normalize=True) * 100
print(cancelled_percentage)

# Percentage of cancellations among repeating guests
repeated_cancelled = df[df['repeated_guest'] == 1]['booking_status'].value_counts(normalize=True) * 100
print(repeated_cancelled)

# Special requests vs booking status
plt.figure(figsize=(10, 6))
sns.boxplot(x='booking_status', y='no_of_special_requests', data=df)
plt.title('Special Requests vs Booking Status')
plt.xlabel('Booking Status')
plt.ylabel('Number of Special Requests')
plt.show()


## Data Preprocessing

- Missing value treatment (if needed)
- Feature engineering (if needed)
- Outlier detection and treatment (if needed)
- Preparing data for modeling
- Any other preprocessing steps (if needed)

In [ ]:
# Data Preprocessing
# Mapping 'booking_status' to binary values
df['booking_status'] = df['booking_status'].map({'Not_Canceled': 0, 'Canceled': 1})

# Verify the conversion
print(df['booking_status'].value_counts())

# Creating new features: total_nights and total_guests
df['total_nights'] = df['no_of_weekend_nights'] + df['no_of_week_nights']
df['total_guests'] = df['no_of_adults'] + df['no_of_children']

# Dropping the original columns to avoid multicollinearity
df.drop(columns=['no_of_weekend_nights', 'no_of_week_nights', 'no_of_adults', 'no_of_children'], inplace=True)

# Encoding other categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Displaying the columns to verify the encoding
print(df_encoded.columns)

# Dropping the 'Booking_ID' column and 'booking_status'
X = df_encoded.drop(columns=['Booking_ID', 'booking_status'])
y = df_encoded['booking_status']

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


## EDA

- It is a good idea to explore the data once again after manipulating it.

In [ ]:
# Distribution of Booking Status after preprocessing
plt.figure(figsize=(8, 6))
sns.countplot(x='booking_status', data=df_encoded)
plt.title('Distribution of Booking Status After Preprocessing')
plt.xlabel('Booking Status')
plt.ylabel('Count')
plt.show()

## Checking Multicollinearity

- In order to make statistical inferences from a logistic regression model, it is important to ensure that there is no multicollinearity present in the data.

In [ ]:
# Checking for multicollinearity using correlation matrix
corr_matrix = df_encoded.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## Building a Logistic Regression model

In [ ]:
# Building the logistic regression model
log_model = sm.Logit(y_train, X_train).fit()
print(log_model.summary())


## Model performance evaluation

In [ ]:
# Predicting and evaluating the logistic regression model
y_pred_log = log_model.predict(X_test) > 0.5
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))
print(confusion_matrix(y_test, y_pred_log))

## Final Model Summary

## Building a Decision Tree model

In [ ]:
# Building the decision tree model
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

# Predicting and evaluating the decision tree model
y_pred_tree = tree_model.predict(X_test)
print('Decision Tree Accuracy:', accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))
print(confusion_matrix(y_test, y_pred_tree))

## Do we need to prune the tree?

## Model Performance Comparison and Conclusions

In [ ]:
# Model performance comparison
log_acc = accuracy_score(y_test, y_pred_log)
tree_acc = accuracy_score(y_test, y_pred_tree)
print(f'Logistic Regression Accuracy: {log_acc}')
print(f'Decision Tree Accuracy: {tree_acc}')


## Actionable Insights and Recommendations

- What profitable policies for cancellations and refunds can the hotel adopt?
- What other recommedations would you suggest to the hotel?

In [ ]:
# Recommendations based on the analysis
recommendations = """
1. Implement a stricter cancellation policy for bookings with longer lead times to mitigate potential revenue loss.
2. Offer incentives or discounts for bookings from market segments with lower cancellation rates.
3. Provide personalized offers or loyalty programs for repeated guests to encourage fewer cancellations.
4. Monitor and adjust room prices dynamically to balance demand and reduce last-minute cancellations.
5. Increase awareness and options for special requests to improve customer satisfaction and reduce cancellations.
"""
print(recommendations)